# Lap Coach Debug Notebook

Quick exploration of MCAP lap data and processing outputs.

In [1]:
import numpy as np
from backend.parser import LapDataParser, _arc_length
from backend.brake_analysis import detect_brake_events, match_brake_events, print_brake_recommendations
from backend.gas_analysis import (
    detect_throttle_applications,
    detect_throttle_plateaus,
    match_throttle_applications,
    match_throttle_plateaus,
    print_gas_recommendations,
)
from backend.steering_analysis import detect_corners, match_corners, print_steering_recommendations


In [2]:
# Load laps
fast_path = "data/hackathon/hackathon_fast_laps.mcap"
good_path = "data/hackathon/hackathon_good_lap.mcap"

fast_states = LapDataParser(fast_path).get_lap_data()
good_states = LapDataParser(good_path).get_lap_data()

print(f"Fast lap samples: {len(fast_states)}")
print(f"Good lap samples: {len(good_states)}")


Fast lap samples: 7427
Good lap samples: 8127


In [3]:
# Build arc-length (metres along track)
fast_arc = _arc_length(fast_states)
good_arc = _arc_length(good_states)

print(f"Fast lap length: {fast_arc[-1]:.1f} m")
print(f"Good lap length: {good_arc[-1]:.1f} m")


Fast lap length: 3450.7 m
Good lap length: 3423.8 m


In [4]:
# Raw channel quick look
def describe_channel(states, name):
    arr = np.array([getattr(s, name) for s in states], dtype=float)
    return dict(min=float(arr.min()), max=float(arr.max()), mean=float(arr.mean()))

for ch in ["steering", "brake", "gas", "speed"]:
    print(ch, "fast", describe_channel(fast_states, ch))
    print(ch, "good", describe_channel(good_states, ch))


steering fast {'min': -0.05778490751981735, 'max': 0.15569822490215302, 'mean': 0.015230424696273857}
steering good {'min': -0.04975922405719757, 'max': 0.1597110629081726, 'mean': 0.01521661134235725}
brake fast {'min': 0.0, 'max': 3717500.0, 'mean': 388035.54598088056}
brake good {'min': 0.0, 'max': 3575000.0, 'mean': 371874.9230958533}
gas fast {'min': 0.0, 'max': 0.9900000095367432, 'mean': 0.6942461336206046}
gas good {'min': 0.004000000189989805, 'max': 0.9900000095367432, 'mean': 0.6359163317899275}
speed fast {'min': 15.031910807253595, 'max': 68.24381873500963, 'mean': 46.124836003408305}
speed good {'min': 12.977169569480054, 'max': 68.37652097798751, 'mean': 41.871269436565655}


In [5]:
# Brake analysis
fast_brakes = detect_brake_events(fast_states, fast_arc)
good_brakes = detect_brake_events(good_states, good_arc)
print(f"Fast brake zones: {len(fast_brakes)}")
print(f"Good brake zones: {len(good_brakes)}")

brake_matches = match_brake_events(fast_brakes, good_brakes)
print_brake_recommendations(brake_matches)


Fast brake zones: 4
Good brake zones: 3

  BRAKES — 4 reference braking zones analysed

  Zone ~416 m  |  entry ⚠️   peak ✅  release ⚠️ 
    • [~416 m] Brake 15 m earlier — you hit the brakes at 367 m, reference brakes at 352 m. Late braking costs you corner entry speed.
    • [~416 m] You hold brakes 18 m past the reference exit. Braking too deep into the corner tightens the radius and delays throttle application on exit.

  Zone ~1439 m  |  entry ⚠️   peak ✅  release ⚠️ 
    • [~1439 m] Brake 13 m earlier — you hit the brakes at 1380 m, reference brakes at 1366 m. Late braking costs you corner entry speed.
    • [~1439 m] You hold brakes 19 m past the reference exit. Braking too deep into the corner tightens the radius and delays throttle application on exit.

  Zone ~2638 m  |  entry ⚠️   peak ✅  release ⚠️ 
    • [~2638 m] Brake 10 m earlier — you hit the brakes at 2560 m, reference brakes at 2550 m. Late braking costs you corner entry speed.
    • [~2638 m] You hold brakes 26 m pa

In [6]:
# Gas analysis
fast_apps = detect_throttle_applications(fast_states, fast_arc)
good_apps = detect_throttle_applications(good_states, good_arc)

fast_plats = detect_throttle_plateaus(fast_states, fast_arc)
good_plats = detect_throttle_plateaus(good_states, good_arc)

app_matches = match_throttle_applications(fast_apps, good_apps)
plat_matches = match_throttle_plateaus(fast_plats, good_plats)

print_gas_recommendations(app_matches, plat_matches)



  GAS — Throttle Application Timing
  ✅  At ~459 m your throttle timing is good (offset +5.3 m).  Reference peak: 65%, yours: 65%.
  ⚠️   At ~1468 m you should get on the throttle sooner — you opened it 47.0 m too late (you: 1515 m, reference: 1468 m).  Getting on gas earlier here will carry more exit speed.
  ⚠️   At ~2692 m you should get on the throttle sooner — you opened it 18.3 m too late (you: 2710 m, reference: 2692 m).  Getting on gas earlier here will carry more exit speed.
  ❌  At ~3428 m the reference driver applies throttle (peaks at 64%) — no matching throttle application found within 60 m of this position.

  GAS — Full-Throttle Plateau Comparison
  ⚠️   Between 0–324 m: you opened throttle 35 m late (get on gas at ~0 m, not 35 m); you lifted 22 m before the reference (held 302 m vs 324 m) — trust the grip and stay on it longer.
  ⚠️   Between 526–716 m: you opened throttle 14 m late (get on gas at ~526 m, not 540 m); you lifted 159 m before the reference (held 31 m vs 

In [7]:
# Steering analysis
fast_corners = detect_corners(fast_states, fast_arc)
good_corners = detect_corners(good_states, good_arc)
print(f"Fast corners: {len(fast_corners)}")
print(f"Good corners: {len(good_corners)}")

fast_steer = np.array([s.steering for s in fast_states], dtype=float)
good_steer = np.array([s.steering for s in good_states], dtype=float)

corner_matches = match_corners(
    fast_corners, good_corners,
    fast_arc, good_arc,
    fast_steer, good_steer,
)
print_steering_recommendations(corner_matches)


Fast corners: 2
Good corners: 2

  STEERING — 2 reference corners analysed

  Corner ←  apex ~1526 m  |  turn-in ⚠️   apex-angle ✅  unwind ⚠️ 
    • [~1526 m (left)] Turn in 31.0 m earlier — you turned at 1531 m, reference at 1500 m. Late turn-in forces a tighter radius and kills exit speed.
    • [~1526 m (left)] You're holding lock 27.2 m longer than the reference. A late unwind tightens the exit line — try to feed out the steering sooner.
    • [~1526 m (left)] Waveform correlation suggests your entire steering input is shifted +19.0 m relative to the reference (positive = you act later on track).

  Corner ←  apex ~2723 m  |  turn-in ⚠️   apex-angle ✅  unwind ⚠️ 
    • [~2723 m (left)] Turn in 27.8 m earlier — you turned at 2729 m, reference at 2701 m. Late turn-in forces a tighter radius and kills exit speed.
    • [~2723 m (left)] You're holding lock 25.7 m longer than the reference. A late unwind tightens the exit line — try to feed out the steering sooner.
    • [~2723 m (left)